# Requirement 4 — Slightly Non-Stationary Environment, Multiple Campaigns

**Reference lab:** `Practical/10_nonstationary_bandits.ipynb` — cell 23 (SW-UCB) and cell 46 (CUSUM-UCB), extended from a plain K-armed bandit to the combinatorial + budget + conflict-graph setting of Requirement 2.

## Provenance — reconciling three team proposals
This is the definitive version, built by reconciling independent work from three places in the team's repository:
- **Requirements 1/2** (`utils/agents.py`'s `CombinatorialUCBAgent`) are taken from **`main`**, the team's agreed definitive R1/R2 branch — already fixed (empirical-mean-cost budget LP, no premature greedy fallback).
- **Requirement 3** (`AdversarialMultiCampaignEnv`, `PrimalDualMultiCampaignAgent`, `req3_config.py`, `run_req3.py`) is taken from **`giovanni`**'s latest commit, the team's agreed definitive R3 branch — including `PrimalDualMultiCampaignAgent`'s `budget_pacing` feature.
- **The benchmark methodology** follows a third proposal (`rec4-partial` branch, `docs/Req4_Linear_Regret_Baseline.tex` and `docs/Req4_Baseline_Code_Fix_Practical.tex`): the dynamic/"prophet" clairvoyant used in earlier drafts inflates regret with a term that is **linear in $T$ regardless of learner quality** (a Jensen-gap "information premium" from knowing every realised $m_t$ in advance — see the derivation in the linked note). The **piecewise expected clairvoyant** is adopted as the primary diagnostic instead.

An earlier draft's self-contained `FixedCombinatorialUCBAgent` workaround (needed because `giovanni`'s own copy of `CombinatorialUCBAgent` had regressed to the original buggy fallback) is no longer needed: `SlidingWindowCombinatorialUCBAgent` / `CUSUMCombinatorialUCBAgent` below subclass **`main`'s** `CombinatorialUCBAgent` directly.

## Environment
`AdversarialMultiCampaignEnv(mode='shocks')` — the **same class** used for Requirement 3, parameterised for FEW, LONG intervals (`block_size=2000` → 5 intervals over `T=10000`) instead of Requirement 3's every-round `'drift'`. Matches the project spec verbatim (p.18). Parameters (`utils/req4_config.py`, built on `utils/req3_config.py`): `VALUES=[0.8,0.8,0.9,0.9]`, `T=10000`, `BUDGET=1600` (rho=0.16), `CONFLICT_EDGES=[(0,1),(2,3)]`.

## Benchmarks — three, reported together
1. **Piecewise expected clairvoyant (PRIMARY)** — an oracle that knows the interval boundaries and each interval's *true* Beta distribution (analytically, from the parameters that generated it — `env.piecewise_win_probabilities()`), but **not** the realised competing bids. This is the natural benchmark for a piecewise-stationary environment, and the one Sliding-Window/CUSUM actually have literature tracking guarantees against (e.g. Garivier & Moulines 2011 for SW-UCB).
2. **OPT$^A$ (secondary)** — best single FIXED distribution for the *whole* horizon, computed from `env.empirical_win_probabilities()`. Same methodology as `run_req3.py`'s own Primal-Dual experiment, kept for continuity; also Primal-Dual's own provable-regret benchmark.
3. **Dynamic / prophet clairvoyant (reference upper bound only)** — knows every realised $m_t$ in advance. Reported for completeness, **not** used to judge learner quality (see Provenance above).

## Three strategies on the same environment (project spec, "Compare")
1. **`SlidingWindowCombinatorialUCBAgent`** — trailing window `W=block_size=2000`, tuned to the interval length rather than the textbook `W=2*sqrt(T)≈200` (with `sum(K_i)=38` cells, the textbook default forgets under-sampled cells before they are well estimated).
2. **`CUSUMCombinatorialUCBAgent`** — per-(campaign,bid) CUSUM change detector on the **win indicator**.
3. **`PrimalDualMultiCampaignAgent`** — Requirement 3's agent, run with `budget_pacing=True` (matches `run_req3.py`'s definitive configuration) and `ogd_eta=0.017`, re-tuned (not assumed) for this environment's 'shocks' mode — a sweep found the same value that Requirement 3 independently tuned for 'drift'.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
if os.path.basename(os.getcwd()) == "deliverables":
    os.chdir(os.path.dirname(os.getcwd()))
print("Working directory:", os.getcwd())

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s")

from utils.run_req4 import run_req4
results = run_req4()

In [ ]:
from IPython.display import Image, display
for f in ["r4/req4_regret_piecewise.png", "r4/req4_regret_opta.png", "r4/req4_regret_prophet.png",
          "r4/req4_budget.png", "r4/req4_cusum_resets.png", "r4/req4_lambda.png"]:
    display(Image(filename=f"outputs/{f}"))

## Results (mean over 20 trials, $T=10000$, $B=1600$)

| Agent | Regret vs Piecewise (primary) | Regret vs OPT$^A$ (secondary) | Regret vs Prophet (reference) | Cumulative cost | Budget used |
|---|---|---|---|---|---|
| **CUSUM Combinatorial-UCB** | **1443.90** (best) | **644.55** (best) | **2717.49** (best) | 1598.91 / 1600 | 99.9% |
| Sliding-Window Combinatorial-UCB | 1532.01 | 732.66 | 2805.60 | 1595.52 / 1600 | 99.7% |
| Primal-Dual (Req 3, `budget_pacing=True`) | 1777.40 | 978.05 | 3050.99 | 1193.92 / 1600 | 74.6% |

Mean CUSUM resets per trial: 55.4 (across `sum(K_i)=38` cells and 4 true regime boundaries per trial).

## Discussion for the presentation

- **The ranking is identical under all three benchmarks**: CUSUM < Sliding-Window < Primal-Dual, regardless of which oracle is used to score regret. This is reassuring — the qualitative conclusion ("Combinatorial-UCB variants track this non-stationary regime better than the reused Primal-Dual agent") does not depend on a benchmark choice that could itself be debated.

- **The benchmark choice changes the *scale* dramatically, and that scale change is itself informative.** Moving from the prophet (2717–3051) to the piecewise-expected benchmark (1444–1777) roughly *halves* every agent's regret — direct empirical confirmation of the Jensen-gap argument in `docs/Req4_Linear_Regret_Baseline.tex`: a large share of the "regret" against the prophet was never a learning failure, just the unavoidable cost of not knowing the realised $m_t$ in advance. OPT$^A$ shrinks it further still (644–978), since it is the weakest of the three oracles (no interval-awareness at all). Reporting only the prophet number, as an earlier draft of this notebook did, would have overstated how far these algorithms are from "good" behaviour.

- **The mechanism behind Primal-Dual's gap is budget under-utilisation**, confirmed independent of benchmark choice: it spends only 74.6% of the budget (1193.92/1600) against 99.7–99.9% for the two Combinatorial-UCB variants. `budget_pacing=True` was enabled specifically to fix this (it targets exactly this failure mode by making the dual OGD target adapt to the realised remaining budget/rounds), and it *does* work at short horizons (a $T{=}500$ smoke test lifted budget use from ~74% to ~99%) — but at the full $T{=}10000$ scale it only nudges usage from 74.2% (unpaced) to 74.6%. Per-interval spend actually overshoots in the first interval, then declines monotonically for the rest of the horizon instead of self-correcting, and $\lambda_t$ stays elevated (~0.7–0.8) for the whole run rather than relaxing once ahead of pace. This is an unresolved, genuinely unexpected result worth presenting as-is.

- **Even the primary (piecewise-expected) regret curve is not visibly sublinear** at this horizon (`req4_regret_piecewise.png`) — a further, honest observation: removing the prophet's artificial linear premium does not by itself guarantee a concave curve. With only 5 intervals of 2000 rounds each, a per-round tracking cost that does not fully vanish *within* one interval (before the next shock arrives) can still look close to linear in aggregate. This is expected for a piecewise-stationary regime with a horizon this short relative to the learning time within each segment, not evidence the benchmark fix failed.

- The CUSUM reset count (55.4/trial across ~4 true regime boundaries and up to 38 cells) is consistent with the detector genuinely reacting to shifts rather than firing on pure noise, though a direct timestamp-level check (comparing reset times against the true regime boundaries, done in an earlier draft of this analysis) shows only a modest, not overwhelming, statistical bias toward firing near the true changes.